#### Celda 1: Carga de datos simulados y reglas de la Data Clean Room (Ya precargado)

In [ ]:
%pip install pandas

In [ ]:
import hashlib
import numpy as np
import pandas as pd

# Fijamos semilla para que todos obtengan exactamente el mismo resultado
np.random.seed(42)
n_usuarios = 500


# Función de seudonimización (Hashing SHA-256)
def hashear_id(email):
  return hashlib.sha256(email.strip().lower().encode()).hexdigest()[:16]


emails = [f"usuario_{i}@ejemplo.com" for i in range(1, n_usuarios + 1)]
hashes = [hashear_id(e) for e in emails]

# --- DATOS EMPRESA A: StreamMusic (Plataforma de Audio) ---
generos = ["Pop", "Rock", "Reggaeton", "Electrónica", "Indie"]
df_music = pd.DataFrame({
    "user_hash": hashes,
    "genero_favorito": np.random.choice(
        generos, size=n_usuarios, p=[0.30, 0.20, 0.25, 0.15, 0.10]
    ),
    "horas_mes": np.random.randint(5, 80, size=n_usuarios),
})

# --- DATOS EMPRESA B: FitWear (Tienda de Ropa Deportiva) ---
# Simulamos que los fans de Electrónica y Reggaeton compran más ropa deportiva
gastos = []
for g in df_music["genero_favorito"]:
  if g == "Electrónica":
    gastos.append(round(np.random.normal(145, 25), 2))
  elif g == "Reggaeton":
    gastos.append(round(np.random.normal(115, 20), 2))
  else:
    gastos.append(round(np.random.normal(65, 15), 2))

df_retail = pd.DataFrame({
    "user_hash": hashes,
    "gasto_eur": np.maximum(15, gastos),  # Mínimo 15€
    "es_recurrente": np.random.choice([True, False], size=n_usuarios),
})


# --- MOTOR DE LA DATA CLEAN ROOM (Árbitro Neutral) ---
class DataCleanRoom:

  def __init__(self, tabla_a, tabla_b, min_k=15):
    # Los datos originales quedan encapsulados (privados)
    self.__tabla_a = tabla_a.copy()
    self.__tabla_b = tabla_b.copy()
    self.min_k = min_k  # Umbral mínimo de privacidad (k-anonimato)

  def ver_datos_crudos(self):
    raise PermissionError(
        "⛔ ACCESO DENEGADO: Una Data Clean Room nunca expone filas"
        " individuales."
    )

  def consultar_agregado(self, agrupar_por, filtro=None):
    # 1. Cruce interno seguro mediante el hash anónimo
    cruce = pd.merge(self.__tabla_a, self.__tabla_b, on="user_hash")

    # 2. Aplicar filtro si el usuario lo solicita
    if filtro:
      cruce = cruce.query(filtro)

    # 3. Validación de privacidad (Evitar reidentificación en grupos pequeños)
    if len(cruce) < self.min_k:
      raise ValueError(
          f"⛔ BLOQUEO DE PRIVACIDAD: El segmento tiene {len(cruce)} usuarios "
          f"(mínimo permitido: {self.min_k})."
      )

    # 4. Devolver únicamente métricas agregadas (sin IDs ni filas)
    resultado = (
        cruce.groupby(agrupar_por)
        .agg(
            usuarios_coincidentes=("user_hash", "count"),
            gasto_medio_eur=("gasto_eur", "mean"),
            horas_medias_musica=("horas_mes", "mean"),
        )
        .round(2)
    )

    # Filtramos también grupos individuales pequeños dentro del groupby
    resultado = resultado[resultado["usuarios_coincidentes"] >= self.min_k]
    return resultado.sort_values(by="gasto_medio_eur", ascending=False)


# Inicializamos la Clean Room con los datos de ambas empresas
dcr = DataCleanRoom(df_music, df_retail, min_k=15)
print(
    "✅ Data Clean Room inicializada correctamente. Tablas cifradas y"
    " listas."
)

### Celda 2: Ejemplo de qué ocurre si intentas saltarte las reglas (Ya precargado)

In [ ]:
# Prueba a ejecutar esta celda para ver cómo la DCR protege la privacidad:
try:
  dcr.ver_datos_crudos()
except PermissionError as e:
  print(e)

try:
  # Intentamos aislar a un grupo muy específico (ataque de inferencia)
  dcr.consultar_agregado(
      agrupar_por="genero_favorito",
      filtro="horas_mes > 78 and gasto_eur > 170",
  )
except ValueError as e:
  print(e)

### Celda 3: Ejercicio 

Completa el código de la siguiente celda para realizar la consulta agregada en la Data Clean Room. 

Para pasar los parámetros a la función, asegúrate de escribirlos exactamente con esta sintaxis (usando argumentos nombrados sin espacios alrededor del `=`):

* `agrupar_por="nombre_columna"` (o una lista si son varias: `agrupar_por=["col1", "col2"]`)
* `filtro="condicion"`

**Ejemplo de uso:**
```python
resultado = ejecutar_consulta(agrupar_por="ciudad", filtro="edad >= 18")
```
*(Si los nombres de la función o los ejemplos de las columnas en tu notebook son distintos, solo tienes que cambiar esa línea de ejemplo).*

In [ ]:
# ==============================================================================
# 🎯 TU RETO (3 - 4 líneas de código):
# 1. Usa dcr.consultar_agregado(...) para cruzar los datos agrupando por "genero_favorito".
# 2. Filtra opcionalmente por usuarios recurrentes (filtro="es_recurrente == True") o analízalos todos.
# 3. Muestra la tabla resultante e identifica qué género musical tiene mayor gasto medio.
# ==============================================================================

# Escribe tu código aquí abajo:

resultado = dcr.consultar_agregado(
    agrupar_por="genero_favorito", filtro="es_recurrente == True"
)
print(resultado)
top_genero = resultado.index[0]
print(
    f"\n🚀 Insight: El público ideal para la campaña de FitWear escucha"
    f" {top_genero}."
)

### Celda 4: Ejercicio 2

Ahora vamos a ver la verdadera potencia de una Data Clean Room: **cruzar a la vez información de las dos empresas** para encontrar segmentos muy concretos sin que ninguna vea los datos de la otra.

Queremos analizar solo a los **oyentes muy activos** (los que escuchan 40 horas o más al mes en StreamMusic) y ver, dentro de cada género musical, si gastan más los clientes nuevos o los recurrentes de FitWear. Además, verás cómo la Clean Room **oculta automáticamente** los grupos demasiado pequeños (menos de 15 personas) para proteger su privacidad.

In [ ]:
# ==============================================================================
# 🎯 TU RETO 2 (3 - 4 líneas de código):
# 1. Usa dcr.consultar_agregado(...) agrupando por ["genero_favorito", "es_recurrente"].
# 2. Filtra para quedarte solo con los oyentes muy activos (filtro="horas_mes >= 40").
# 3. Muestra la tabla resultante y comprueba qué segmento lidera el gasto y qué grupo pequeño ha sido ocultado por privacidad.
# ==============================================================================

# Escribe tu código aquí abajo:

resultado_2 = dcr.consultar_agregado(
    agrupar_por=["genero_favorito", "es_recurrente"],
    filtro="horas_mes >= 40",
)
print(resultado_2)
top_genero, es_rec = resultado_2.index[0]
print(
    f"\n🚀 Insight: Entre los oyentes más activos, el segmento más rentable es"
    f" {top_genero} (recurrente={es_rec})."
)